In [36]:
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.compute as pc
import pyarrow.parquet as pq
import pyarrow.csv as pv
import pandas as pd


import s3fs
import pandas
import geopandas
from pathlib import Path

import utils.S3hsclient as hsclient

from tqdm.notebook import tqdm

In [4]:
root_dir = Path('stream-v1-data-original')
pq_dir = Path('stream-v1-data-parquet')
pq_dir.mkdir(exist_ok=True)

## Create Metadata Parquet

In [5]:
df = pandas.read_csv(root_dir / "01_metadata/metadata.csv")
df.to_parquet(pq_dir / 'metadata.parquet')

## Create Static Catchment Characteristics Parquet

In [14]:
df = pandas.read_csv(root_dir / "04_static_catchment_characteristics/static_catchment_characteristics.csv")
df.to_parquet(pq_dir / 'static_catchment_characteristics.parquet')

## Create Watersheds Parquet

In [30]:
# loop through all watershed shapefiles and combine
# them into a single geodataframe.
gdfs = []
shapefiles = list((root_dir / "03_shapefiles").glob('*.shp'))
for shapefile in tqdm(shapefiles, desc="Loading Shapefiles", unit="file"):
    gdfs.append(geopandas.read_file(shapefile))
gdf = pandas.concat(gdfs)

# save the watersheds geodataframe to parquet
gdf.to_parquet(pq_dir / 'watersheds.parquet', index=False)

Loading Shapefiles:   0%|          | 0/4268 [00:00<?, ?file/s]

## Create LULC, Anthropogenic, Grab Sample, and Water Quality Parquets

In [10]:
def create_parquet(input_data_dir, output_filename,
                   date_cols = [], sort_by = None):
    """
    Loops through all of the CSV files in a given directory, loads them
    into a Pandas object, then saves the result to Parquet.
    """
    
    input_data_dir = Path(input_data_dir)
    dfs = []

    # loop through all of the csv files in the current directory
    # and build a list of Pandas DataFrames. Then concatenate
    # the list of DataFrames into a single object.
    for f in input_data_dir.glob('*.csv'):
        df_temp = pandas.read_csv(f,
                                 parse_dates=date_cols)
        df_temp['STREAM_ID'] = f.stem
        dfs.append(df_temp)
    df = pandas.concat(dfs)

    # sort the values in the DataFrame using the sort_by
    # input argument. This is used to improve query performance
    if sort_by is not None:
        df.sort_values(by=sort_by, inplace=True)

    # save the result as a parquet file in the user-specified
    # output location.
    df.to_parquet(output_filename, index=False)
    

In [11]:
create_parquet(root_dir / '06_dynamic_lulc',
               pq_dir / 'dynamic_lulc.parquet',
               date_cols=['DateTime'],
               sort_by=['STREAM_ID', 'DateTime'])

In [12]:
create_parquet(root_dir / '07_dynamic_anthropogenic',
               pq_dir / 'dynamic_anthropogenic.parquet',
               date_cols=['DateTime'],
               sort_by=['STREAM_ID', 'DateTime'])

In [13]:
create_parquet(root_dir / '09_grab_samples',
               pq_dir / 'grab_samples.parquet',
               date_cols=['DateTime'],
               sort_by=['STREAM_ID', 'DateTime'])

## Create Streamflow Discharge Parquet

In [40]:
# load streamflow data into pandas
streamflow_paths = list((root_dir / "08_streamflow_discharge").glob('*.csv'))

dfs = []
for f in tqdm(streamflow_paths, desc="Loading Streamflow", unit="file"):
    df_temp = pandas.read_csv(f, parse_dates=['DateTime'])
    df_temp['STREAM_ID'] = f.stem
    dfs.append(df_temp)
df = pandas.concat(dfs, ignore_index=True)

Loading Streamflow:   0%|          | 0/2186 [00:00<?, ?file/s]

In [43]:
# convert pandas to pyarrow table
table = pa.Table.from_pandas(df, preserve_index=False)

# save table using 'hive' partitioning
parquet_format = ds.ParquetFileFormat()
write_options = parquet_format.make_write_options(
    compression="zstd",
)

partitioning = ds.partitioning(
    pa.schema([
        ("STREAM_ID", pa.large_string()),
    ]),
    flavor="hive"  # This creates gauge=X/year=Y/ paths
)

ds.write_dataset(
    table,
    base_dir=pq_dir / 'streamflow_discharge',
    format=parquet_format,
    partitioning=partitioning,  
    existing_data_behavior="overwrite_or_ignore",
    max_rows_per_file=5_000_000,
    max_rows_per_group=250_000,
)